# Step 08 — Fast fallback annotation from ResolVI-corrected UCell signatures

This notebook implements a deliberately simple, cell-level fallback annotation
that does **not** depend on the cross-cancer atlas or cluster transfer.

It reads, for every sample:

```text
03C raw-count H5AD + existing corrected-expression UCell scores
03A authoritative all-cell ResolVI-corrected Zarr
```

and applies a transparent hierarchical decision tree.

## Hierarchy

### Level 1 — broad lineage

A cell is called T lineage only when:

```text
T-core UCell is sample-high
AND
T-core evidence exceeds the competing non-T lineage programs
```

Remaining cells are assigned by the strongest supported lineage program:

```text
Tumor
Endothelial
Monocyte/macrophage
Fibroblast
B/plasma
NK
Other/unresolved
```

`Tumor` is a composite of:

```text
epithelial/keratin
melanocytic melanoma
dedifferentiated melanoma
```

### Level 2 — T-cell subtype

Only within cells already called T lineage:

```text
Treg has first priority when the conservative Treg-core signature is strong
CD8+ uses a CD8B-free cytotoxic-T signature
CD4+ uses the existing corrected CD4/helper signature
ambiguous/unspecified T cells remain explicitly labeled
```

The final labels are:

```text
Tcell:Treg
Tcell:CD8+
Tcell:CD4+
Tcell:CD4_CD8_ambiguous
Tcell:unspecified
Tumor
Endothelial
Monocyte_macrophage
Fibroblast
B_plasma
NK
Other_unresolved
```

## Existing versus newly calculated scores

For speed, the notebook reuses existing Step 03C UCell columns whenever their
signature is still appropriate.

It freshly calculates only the refined signatures that are needed:

```text
CD8_noCD8B
Treg_core
NK_specific
myeloid_macrophage_specific
fibroblast_collagen
```

If an expected existing 03C score is absent, it is also calculated from the
03A corrected Zarr.

## Treg safeguard

The Treg call is sample-adaptive and requires:

```text
T lineage
strong Treg-core UCell score
CD4-compatible signal
Treg evidence above the CD8/cytotoxic program
```

The raw-count matrix is used only to add an independent audit column:

```text
fallback_Treg_raw_anchor
```

By default, raw support is **not required**, preserving the requested
signature-only method. The notebook separately reports:

```text
Tcell:Treg with raw anchor
Tcell:Treg signature-only
```

and warns when the inferred Treg fraction is unusually high.

## Matrix convention

The saved H5AD files keep:

```python
adata.X
# original sparse integer raw counts from Step 03C

adata.obs
# UCell scores, thresholds, labels, and confidence fields
```

The dense corrected matrix is not duplicated. Its 03A Zarr path is recorded in
`.uns`.

## Optional score-space Leiden

The primary result is the hierarchical cell-level annotation. A lightweight
Leiden clustering on the standardized signature-score matrix is available for
diagnostics but disabled by default.

## v2 recovery behavior

The original notebook wrote each sample's metadata Parquet, thresholds, counts,
and figures **before** the failing H5AD serialization step. This version detects
those completed Parquets and restores the fallback scores/labels into the
original Step 03C H5AD, avoiding another pyUCell run.

The failure was caused by a nested `list[dict]` inside:

```python
adata.uns["fallback_resolvi_ucell_annotation"]["scoring_info"]
```

AnnData/HDF5 interpreted that heterogeneous object array as a string dataset and
raised:

```text
TypeError: Can't implicitly convert non-string objects to strings
```

v2 stores compact scalar provenance plus a JSON string instead.


In [1]:
# ---------------------------------------------------------------------
# Environment selection — run before importing pyUCell/Torch
# ---------------------------------------------------------------------
import os

GPU_ID = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ.setdefault("OMP_NUM_THREADS", "16")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "16")
os.environ.setdefault("MKL_NUM_THREADS", "16")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "16")

print("CUDA_VISIBLE_DEVICES:", GPU_ID)

CUDA_VISIBLE_DEVICES: 0


In [2]:
# ---------------------------------------------------------------------
# Imports and pyUCell compatibility inspection
# ---------------------------------------------------------------------
from __future__ import annotations

import gc
import hashlib
import inspect
import json
import math
import time
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import zarr

try:
    import pyucell as uc
    import importlib.metadata

    PYUCELL_AVAILABLE = True
    PYUCELL_VERSION = importlib.metadata.version("pyucell")
    UCELL_SIGNATURE = inspect.signature(
        uc.compute_ucell_scores
    )
    UCELL_PARAMETERS = UCELL_SIGNATURE.parameters
    UCELL_ACCEPTS_VAR_KWARGS = any(
        parameter.kind
        is inspect.Parameter.VAR_KEYWORD
        for parameter in UCELL_PARAMETERS.values()
    )
except Exception as exc:
    uc = None
    PYUCELL_AVAILABLE = False
    PYUCELL_VERSION = None
    UCELL_SIGNATURE = None
    UCELL_PARAMETERS = {}
    UCELL_ACCEPTS_VAR_KWARGS = False
    PYUCELL_IMPORT_ERROR = (
        f"{type(exc).__name__}: {exc}"
    )

print("anndata:", ad.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("pyUCell available:", PYUCELL_AVAILABLE)
if PYUCELL_AVAILABLE:
    print("pyUCell:", PYUCELL_VERSION)
    print("pyUCell signature:", UCELL_SIGNATURE)
else:
    print("pyUCell import error:", PYUCELL_IMPORT_ERROR)

anndata: 0.12.19
numpy: 1.26.4
pandas: 2.3.3
pyUCell available: True
pyUCell: 0.7.3
pyUCell signature: (adata: anndata._core.anndata.AnnData, signatures: dict[str, list[str]], layer: str = None, max_rank: int = 1500, ties_method: str = 'average', missing_genes: str = 'impute', chunk_size: int | None = None, w_neg: float = 1.0, suffix: str = '_UCell', n_jobs: int = -1, device: str | None = 'cpu')


/tmp/ipykernel_86541/3723025631.py:48: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata:", ad.__version__)


In [3]:
# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057"
)
PIPELINE_ROOT = (
    PROJECT_ROOT
    / "tmp"
    / "proseg_resolvi_immune_enrichment_v1"
)

ALLCELL_ZARR_ROOT = (
    PIPELINE_ROOT
    / "03a_resolvi_allcell_zarr"
)
SCORED_ROOT = (
    PIPELINE_ROOT
    / "03c_reference_ucell_rescue"
)

OUTPUT_ROOT = (
    PIPELINE_ROOT
    / "08_fallback_ucell_hierarchical_annotation"
)
OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SAMPLE_INFO = {
    "Screen_39_21": {
        "patient": "patient_39_21",
        "cancer_type": "NSCLC",
        "biopsy_stage": "Screen",
    },
    "C2D15_39_21": {
        "patient": "patient_39_21",
        "cancer_type": "NSCLC",
        "biopsy_stage": "C2D15",
    },
    "Screen_17_26": {
        "patient": "patient_17_26",
        "cancer_type": "NSCLC",
        "biopsy_stage": "Screen",
    },
    "C2D15_17_26": {
        "patient": "patient_17_26",
        "cancer_type": "NSCLC",
        "biopsy_stage": "C2D15",
    },
    "Screen_18_23": {
        "patient": "patient_18_23",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen",
    },
    "C2D15_18_23": {
        "patient": "patient_18_23",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15",
    },
    "Screen_16_22": {
        "patient": "patient_16_22",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen",
    },
    "C2D15_16_22": {
        "patient": "patient_16_22",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15",
    },
    "Screen_30_16": {
        "patient": "patient_30_16",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen",
    },
    "C2D15_30_16": {
        "patient": "patient_30_16",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15",
    },
    "Screen_23_25": {
        "patient": "patient_23_25",
        "cancer_type": "colon_cancer",
        "biopsy_stage": "Screen",
    },
    "C2D15_23_25": {
        "patient": "patient_23_25",
        "cancer_type": "colon_cancer",
        "biopsy_stage": "C2D15",
    },
}

SECTION_NAMES = list(
    SAMPLE_INFO
)

CORRECTED_LAYER = (
    "resolvi_corrected_10k"
)

# Reuse existing 03C scores whenever the old signature is suitable.
USE_EXISTING_03C_SCORES = True

# The refined signatures below are always recalculated unless their fallback
# columns already exist and OVERWRITE_FALLBACK_SCORES=False.
OVERWRITE_FALLBACK_SCORES = False

# pyUCell execution.
PREFER_DEVICE = "cuda:0"
UCELL_MAX_RANK = 1_500
UCELL_OUTER_CELL_CHUNK = 5_000
UCELL_INNER_CHUNK_SIZE = 1_000
UCELL_CPU_N_JOBS = min(
    8,
    os.cpu_count() or 1,
)
UCELL_GPU_FALLBACK_TO_CPU = True

# Broad-lineage thresholds are calculated independently within each sample.
# A score must exceed:
#   fixed floor
#   configured sample quantile
#   median + robust_z × scaled MAD
BROAD_THRESHOLD_CONFIG = {
    "T_core": {
        "quantile": 0.90,
        "robust_z": 1.5,
        "floor": 0.03,
    },
    "Tumor": {
        "quantile": 0.90,
        "robust_z": 1.5,
        "floor": 0.03,
    },
    "Endothelial": {
        "quantile": 0.90,
        "robust_z": 1.5,
        "floor": 0.03,
    },
    "Monocyte_macrophage": {
        "quantile": 0.90,
        "robust_z": 1.5,
        "floor": 0.03,
    },
    "Fibroblast": {
        "quantile": 0.90,
        "robust_z": 1.5,
        "floor": 0.03,
    },
    "B_plasma": {
        "quantile": 0.90,
        "robust_z": 1.5,
        "floor": 0.03,
    },
    "NK": {
        "quantile": 0.90,
        "robust_z": 1.5,
        "floor": 0.03,
    },
}

# T cells are accepted only when the T-core standardized score is higher than
# all competing broad lineages by at least this amount.
T_LINEAGE_Z_MARGIN = 0.25

# For non-T cells, the strongest supported lineage must exceed the runner-up.
NON_T_Z_MARGIN = 0.10

# T-subtype thresholds are calculated within the T-lineage cells in each sample.
CD4_SUBTYPE_QUANTILE = 0.50
CD8_SUBTYPE_QUANTILE = 0.50
SUBTYPE_SCORE_FLOOR = 0.02
CD4_CD8_Z_MARGIN = 0.25

TREG_THRESHOLD_CONFIG = {
    "quantile": 0.975,
    "robust_z": 2.5,
    "floor": 0.03,
}
TREG_VS_CD8_Z_MARGIN = 0.50
TREG_REQUIRE_CD4_COMPATIBILITY = True
USE_RAW_TREG_ANCHOR_FOR_FINAL_LABEL = False

# Emit a warning but do not automatically change labels.
TREG_WARNING_FRACTION_OF_TCELLS = 0.15

# Optional broad labels; they remain competitors even if disabled.
LABEL_B_PLASMA = True
LABEL_NK = True

# Optional diagnostic clustering on signature-score z values.
RUN_SIGNATURE_LEIDEN = False
SIGNATURE_LEIDEN_RESOLUTION = 0.40
SIGNATURE_LEIDEN_N_NEIGHBORS = 15

PLOT_DPI = 500
PLOT_MAX_CELLS = 250_000

WRITE_ANNOTATED_H5AD = True
WRITE_T_ONLY_H5AD = True
WRITE_METADATA_PARQUET = True
WRITE_SPATIAL_PARQUET = True
H5AD_COMPRESSION = "lzf"

CONTINUE_ON_ERROR = True

# The v1 notebook wrote the cell-level fallback Parquet before the H5AD write.
# Reuse that completed work and skip fresh pyUCell scoring when possible.
RESUME_FROM_EXISTING_METADATA = True
REQUIRE_COMPLETE_RESUME_METADATA = True

PIPELINE_VERSION = (
    "2026-08-02-fallback-resolvi-ucell-hierarchy-v2-resume-h5ad-fix"
)

print("Samples:", SECTION_NAMES)
print("Output root:", OUTPUT_ROOT)

Samples: ['Screen_39_21', 'C2D15_39_21', 'Screen_17_26', 'C2D15_17_26', 'Screen_18_23', 'C2D15_18_23', 'Screen_16_22', 'C2D15_16_22', 'Screen_30_16', 'C2D15_30_16', 'Screen_23_25', 'C2D15_23_25']
Output root: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/08_fallback_ucell_hierarchical_annotation


In [4]:
# ---------------------------------------------------------------------
# Signature definitions
# ---------------------------------------------------------------------
# Existing 03C signatures whose score columns can be reused.
STANDARD_SIGNATURES = {
    "T_core": {
        "genes": [
            "CD3D",
            "CD3E",
            "TRAC",
            "TRBC1",
            "TRBC2",
            "CD247",
            "LCK",
            "LAT",
            "CD2",
        ],
        "existing_columns": [
            "T_cell_core_resolvi_UCell",
        ],
    },
    "CD4": {
        "genes": [
            "IL7R",
            "CCR7",
            "TCF7",
            "LEF1",
            "MAL",
            "LTB",
            "CD4",
        ],
        "existing_columns": [
            "CD4_helper_resolvi_UCell",
        ],
    },
    "Endothelial": {
        "genes": [
            "PECAM1",
            "VWF",
            "KDR",
            "ESAM",
            "ENG",
            "EMCN",
            "RAMP2",
            "RGCC",
            "PLVAP",
            "CA4",
        ],
        "existing_columns": [
            "endothelial_resolvi_UCell",
        ],
    },
    "Epithelial": {
        "genes": [
            "EPCAM",
            "TACSTD2",
            "KRT7",
            "KRT8",
            "KRT18",
            "KRT19",
            "MUC1",
            "KRT5",
            "KRT6A",
            "KRT6B",
            "KRT14",
            "KRT17",
            "TP63",
            "SFN",
            "DSG3",
            "CEACAM5",
            "CEACAM6",
            "MSLN",
        ],
        "existing_columns": [
            "epithelial_keratin_resolvi_UCell",
        ],
    },
    "Melanoma_melanocytic": {
        "genes": [
            "MLANA",
            "PMEL",
            "TYR",
            "DCT",
            "MITF",
            "SOX10",
            "S100B",
        ],
        "existing_columns": [
            "melanoma_melanocytic_resolvi_UCell",
        ],
    },
    "Melanoma_dedifferentiated": {
        "genes": [
            "AXL",
            "NGFR",
            "SOX9",
        ],
        "existing_columns": [
            "melanoma_dedifferentiated_resolvi_UCell",
        ],
    },
    "B_cell": {
        "genes": [
            "MS4A1",
            "CD79A",
            "CD79B",
            "CD37",
            "CD74",
            "CD22",
        ],
        "existing_columns": [
            "B_cell_resolvi_UCell",
        ],
    },
    "Plasma_cell": {
        "genes": [
            "MZB1",
            "JCHAIN",
            "SDC1",
            "XBP1",
            "IGHG1",
            "IGKC",
        ],
        "existing_columns": [
            "plasma_cell_resolvi_UCell",
        ],
    },
}

# Refined signatures recalculated from the corrected 03A matrix.
REFINED_SIGNATURES = {
    # CD8B is deliberately absent because its probe is known to underperform.
    "CD8_noCD8B": [
        "CD8A",
        "CTSW",
        "CCL5",
        "CCL4",
        "GZMK",
        "GZMH",
        "PRF1",
        "GZMB",
    ],
    # More specific than the original activated-T-heavy Treg signature.
    "Treg_core": [
        "FOXP3",
        "IL2RA",
        "IKZF2",
        "CCR8",
    ],
    # Excludes TYROBP/FCER1G as decisive markers because they are NK-shared.
    "NK_specific": [
        "KLRD1",
        "GNLY",
        "XCL1",
        "XCL2",
        "KLRF1",
        "NCR1",
        "NCAM1",
    ],
    "Myeloid_macrophage_specific": [
        "LST1",
        "LYZ",
        "AIF1",
        "CTSS",
        "FCGR3A",
        "CSF1R",
        "CD68",
        "C1QA",
        "C1QB",
        "C1QC",
        "APOE",
        "MSR1",
        "MRC1",
        "TREM2",
        "GPNMB",
        "CTSD",
        "LGALS3",
    ],
    "Fibroblast_collagen": [
        "COL1A1",
        "COL1A2",
        "COL3A1",
        "COL5A1",
        "COL5A2",
        "COL6A1",
        "COL6A2",
        "COL6A3",
        "DCN",
        "LUM",
        "PDGFRA",
        "C7",
    ],
}

# Canonical score columns used by the classifier.
CANONICAL_SCORE_COLUMNS = {
    "T_core": "fallback_T_core_UCell",
    "CD4": "fallback_CD4_UCell",
    "CD8": "fallback_CD8_noCD8B_UCell",
    "Treg": "fallback_Treg_core_UCell",
    "NK": "fallback_NK_specific_UCell",
    "Myeloid": (
        "fallback_Myeloid_macrophage_specific_UCell"
    ),
    "Fibroblast": (
        "fallback_Fibroblast_collagen_UCell"
    ),
    "Endothelial": (
        "fallback_Endothelial_UCell"
    ),
    "Epithelial": (
        "fallback_Epithelial_UCell"
    ),
    "Melanoma_melanocytic": (
        "fallback_Melanoma_melanocytic_UCell"
    ),
    "Melanoma_dedifferentiated": (
        "fallback_Melanoma_dedifferentiated_UCell"
    ),
    "B_cell": "fallback_B_cell_UCell",
    "Plasma_cell": "fallback_Plasma_cell_UCell",
}

In [5]:
# ---------------------------------------------------------------------
# Paths, hashing, metadata, and portable H5AD writing
# ---------------------------------------------------------------------
def paths_for_sample(
    sample: str,
) -> dict[str, Path]:
    out = OUTPUT_ROOT / sample
    figures = out / "figures"

    out.mkdir(
        parents=True,
        exist_ok=True,
    )
    figures.mkdir(
        parents=True,
        exist_ok=True,
    )

    return {
        "scored": (
            SCORED_ROOT
            / sample
            / f"{sample}_reference_ucell_scored.h5ad"
        ),
        "zarr": (
            ALLCELL_ZARR_ROOT
            / sample
            / f"{sample}_resolvi_allcells.zarr"
        ),
        "zarr_summary": (
            ALLCELL_ZARR_ROOT
            / sample
            / f"{sample}_resolvi_allcells_zarr_summary.json"
        ),
        "out": out,
        "figures": figures,
        "annotated": (
            out
            / f"{sample}_fallback_ucell_annotated.h5ad"
        ),
        "T_only": (
            out
            / f"{sample}_fallback_Tcells_only.h5ad"
        ),
        "metadata": (
            out
            / f"{sample}_fallback_ucell_metadata.parquet"
        ),
        "spatial": (
            out
            / f"{sample}_fallback_ucell_spatial.parquet"
        ),
        "thresholds": (
            out
            / f"{sample}_fallback_ucell_thresholds.csv"
        ),
        "coverage": (
            out
            / f"{sample}_fallback_signature_gene_coverage.csv"
        ),
        "counts": (
            out
            / f"{sample}_fallback_cell_type_counts.csv"
        ),
        "summary": (
            out
            / f"{sample}_fallback_annotation_summary.json"
        ),
    }


def names_hash(
    values,
) -> str:
    digest = hashlib.sha256()
    for value in values:
        digest.update(
            str(value).encode(
                "utf-8"
            )
        )
        digest.update(b"\0")
    return digest.hexdigest()


def write_json(
    payload,
    path: str | Path,
) -> None:
    path = Path(path)
    temporary = path.with_suffix(
        path.suffix + ".tmp"
    )
    temporary.write_text(
        json.dumps(
            payload,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )
    temporary.replace(path)


def _is_nullable_string_series(
    series: pd.Series,
) -> bool:
    return (
        isinstance(
            series.dtype,
            pd.StringDtype,
        )
        or type(
            series.array
        ).__name__
        in {
            "StringArray",
            "ArrowStringArray",
        }
        or str(
            series.dtype
        ) == "str"
        or str(
            series.dtype
        ).startswith(
            "string"
        )
    )


def _legacy_string_frame(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    output = frame.copy()

    for column in output.columns:
        series = output[column]

        if _is_nullable_string_series(
            series
        ):
            output[column] = (
                series
                .fillna("")
                .astype(str)
                .astype(object)
            )
        elif pd.api.types.is_object_dtype(
            series.dtype
        ):
            # HDF5 cannot write an object array containing dictionaries,
            # lists, Paths, numpy scalars, or mixed Python objects as strings.
            # Convert every object value deterministically.
            def _safe_text(value):
                if value is None or value is pd.NA:
                    return ""
                if isinstance(value, bytes):
                    return value.decode(
                        "utf-8",
                        errors="replace",
                    )
                if isinstance(value, str):
                    return value
                if isinstance(value, np.generic):
                    value = value.item()
                if isinstance(
                    value,
                    (
                        dict,
                        list,
                        tuple,
                        set,
                        np.ndarray,
                        Path,
                    ),
                ):
                    return json.dumps(
                        value,
                        default=str,
                        sort_keys=True,
                    )
                return str(value)

            output[column] = (
                series
                .map(_safe_text)
                .astype(object)
            )

    if (
        str(
            output.index.dtype
        ) == "str"
        or str(
            output.index.dtype
        ).startswith(
            "string"
        )
        or type(
            output.index.array
        ).__name__
        in {
            "StringArray",
            "ArrowStringArray",
        }
    ):
        name = output.index.name
        output.index = pd.Index(
            pd.Series(
                output.index,
                dtype="string",
            )
            .fillna("")
            .astype(str)
            .to_numpy(
                dtype=object
            ),
            name=name,
        )

    return output


def safe_write_h5ad(
    adata_object: ad.AnnData,
    filename: str | Path,
    *,
    compression: str = "lzf",
) -> None:
    filename = Path(filename)
    filename.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    adata_object.obs = (
        _legacy_string_frame(
            adata_object.obs
        )
    )
    adata_object.var = (
        _legacy_string_frame(
            adata_object.var
        )
    )

    temporary = filename.with_name(
        f"{filename.stem}.tmp{filename.suffix}"
    )
    if temporary.exists():
        temporary.unlink()

    with ad.settings.override(
        allow_write_nullable_strings=False
    ):
        adata_object.write_h5ad(
            temporary,
            compression=compression,
            convert_strings_to_categoricals=False,
        )

    temporary.replace(filename)
    print("Saved:", filename)


def choose_spatial_key(
    adata: ad.AnnData,
) -> str | None:
    for key in (
        "X_spatial",
        "spatial",
        "spatial_fullres",
    ):
        if key in adata.obsm:
            array = np.asarray(
                adata.obsm[key]
            )
            if (
                array.ndim == 2
                and array.shape[0]
                == adata.n_obs
                and array.shape[1]
                >= 2
            ):
                return key
    return None


def restore_existing_fallback_metadata(
    adata: ad.AnnData,
    paths: dict[str, Path],
    sample: str,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    dict,
    dict,
]:
    """Restore completed v1 scores/labels from the pre-write Parquet.

    The v1 process wrote this Parquet before adding the problematic nested
    `scoring_info` payload to `.uns` and before writing H5AD. Reusing it avoids
    repeating corrected-Zarr decoding and pyUCell scoring.
    """
    metadata_path = paths["metadata"]

    if not metadata_path.exists():
        raise FileNotFoundError(
            f"Resume metadata is absent: {metadata_path}"
        )

    metadata = pd.read_parquet(
        metadata_path
    )
    if "cell_id" not in metadata.columns:
        raise KeyError(
            f"{metadata_path} has no 'cell_id' column."
        )

    metadata["cell_id"] = (
        metadata["cell_id"]
        .astype(str)
    )
    if metadata["cell_id"].duplicated().any():
        duplicates = metadata.loc[
            metadata["cell_id"].duplicated(
                keep=False
            ),
            "cell_id",
        ].head(10).tolist()
        raise ValueError(
            "Resume metadata contains duplicated cell IDs: "
            f"{duplicates}"
        )

    metadata = metadata.set_index(
        "cell_id"
    )

    missing_cells = (
        adata.obs_names.difference(
            metadata.index
        )
    )
    extra_cells = (
        metadata.index.difference(
            adata.obs_names
        )
    )

    if len(missing_cells):
        raise ValueError(
            f"{sample}: resume metadata is missing "
            f"{len(missing_cells):,} H5AD cells."
        )
    if len(extra_cells):
        warnings.warn(
            f"{sample}: ignoring {len(extra_cells):,} "
            "metadata rows not present in the H5AD."
        )

    aligned = metadata.reindex(
        adata.obs_names
    )

    fallback_columns = [
        column
        for column in aligned.columns
        if column.startswith(
            "fallback_"
        )
    ]
    if not fallback_columns:
        raise ValueError(
            f"{sample}: resume metadata has no fallback_* columns."
        )

    required_columns = {
        "fallback_cell_type",
        "fallback_level1_lineage",
        "fallback_level2_T_subtype",
        "fallback_T_lineage",
        "fallback_Treg_signature",
        "fallback_Treg_raw_anchor",
    }
    missing_required = sorted(
        required_columns
        - set(fallback_columns)
    )
    if (
        missing_required
        and REQUIRE_COMPLETE_RESUME_METADATA
    ):
        raise KeyError(
            f"{sample}: resume metadata is incomplete. "
            f"Missing: {missing_required}"
        )

    categorical_columns = {
        "fallback_cell_type",
        "fallback_level1_lineage",
        "fallback_level2_T_subtype",
        "fallback_dominant_nonT_program",
        "fallback_signature_leiden",
    }

    for column in fallback_columns:
        values = aligned[column]

        if column in categorical_columns:
            adata.obs[column] = pd.Categorical(
                values
                .astype("string")
                .fillna("")
                .astype(str)
            )
        elif pd.api.types.is_bool_dtype(
            values.dtype
        ):
            adata.obs[column] = (
                values
                .fillna(False)
                .to_numpy(
                    dtype=bool
                )
            )
        elif pd.api.types.is_numeric_dtype(
            values.dtype
        ):
            adata.obs[column] = (
                pd.to_numeric(
                    values,
                    errors="coerce",
                )
                .to_numpy()
            )
        else:
            adata.obs[column] = (
                values
                .astype("string")
                .fillna("")
                .astype(str)
                .to_numpy(
                    dtype=object
                )
            )

    for column in (
        "sample",
        "patient",
        "cancer_type",
        "biopsy_stage",
    ):
        if column in aligned.columns:
            adata.obs[column] = (
                aligned[column]
                .astype("string")
                .fillna("")
                .astype(str)
                .to_numpy(
                    dtype=object
                )
            )

    thresholds = (
        pd.read_csv(
            paths["thresholds"]
        )
        if paths["thresholds"].exists()
        else pd.DataFrame()
    )
    coverage = (
        pd.read_csv(
            paths["coverage"]
        )
        if paths["coverage"].exists()
        else pd.DataFrame()
    )

    t_lineage = (
        adata.obs[
            "fallback_T_lineage"
        ]
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )
    treg = (
        adata.obs[
            "fallback_cell_type"
        ]
        .astype(str)
        .eq("Tcell:Treg")
        .to_numpy()
    )
    raw_anchor = (
        adata.obs[
            "fallback_Treg_raw_anchor"
        ]
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )

    t_count = int(
        t_lineage.sum()
    )
    treg_count = int(
        treg.sum()
    )
    treg_fraction = (
        treg_count
        / max(
            t_count,
            1,
        )
    )

    warning_messages = []
    if (
        t_count > 0
        and treg_fraction
        > float(
            TREG_WARNING_FRACTION_OF_TCELLS
        )
    ):
        message = (
            f"{sample}: restored Treg fraction "
            f"{treg_fraction:.1%} exceeds the warning level "
            f"{TREG_WARNING_FRACTION_OF_TCELLS:.1%}."
        )
        warnings.warn(
            message
        )
        warning_messages.append(
            message
        )

    annotation_summary = {
        "n_cells": int(
            adata.n_obs
        ),
        "n_T_lineage": t_count,
        "fraction_T_lineage": float(
            t_lineage.mean()
        ),
        "n_Treg": treg_count,
        "fraction_Treg_of_T": float(
            treg_fraction
        ),
        "n_Treg_with_raw_anchor": int(
            (
                treg
                & raw_anchor
            ).sum()
        ),
        "warnings": warning_messages,
    }

    scoring_info = {
        "resume_used": True,
        "source_metadata_parquet": str(
            metadata_path
        ),
        "n_restored_columns": int(
            len(
                fallback_columns
            )
        ),
        "restored_columns": (
            fallback_columns
        ),
        "fresh_UCell_recomputed": False,
    }

    print(
        f"{sample}: restored {len(fallback_columns):,} "
        "fallback columns from existing Parquet; "
        "fresh UCell scoring was skipped."
    )

    return (
        coverage,
        thresholds,
        annotation_summary,
        scoring_info,
    )


def h5ad_safe_scoring_info(
    scoring_info: dict,
) -> dict:
    """Keep compact scalar provenance in `.uns`; full details stay in JSON."""
    return {
        "resume_used": bool(
            scoring_info.get(
                "resume_used",
                False,
            )
        ),
        "fresh_UCell_recomputed": bool(
            scoring_info.get(
                "fresh_UCell_recomputed",
                bool(
                    scoring_info.get(
                        "fresh_scores",
                        []
                    )
                ),
            )
        ),
        "n_reused_scores": int(
            len(
                scoring_info.get(
                    "reused_scores",
                    {}
                )
            )
        ),
        "n_fresh_scores": int(
            len(
                scoring_info.get(
                    "fresh_scores",
                    []
                )
            )
        ),
        "details_json": json.dumps(
            scoring_info,
            default=str,
            sort_keys=True,
        ),
    }


In [6]:
# ---------------------------------------------------------------------
# pyUCell compatibility and corrected-expression chunk scoring
# ---------------------------------------------------------------------
def _ucell_accepts(
    name: str,
) -> bool:
    return (
        name in UCELL_PARAMETERS
        or UCELL_ACCEPTS_VAR_KWARGS
    )


def map_genes(
    var_names: pd.Index,
    genes: list[str],
) -> list[str]:
    lookup = {}
    for value in var_names.astype(str):
        lookup.setdefault(
            value.upper(),
            value,
        )

    output = []
    for gene in genes:
        mapped = lookup.get(
            str(gene).upper()
        )
        if (
            mapped is not None
            and mapped not in output
        ):
            output.append(mapped)

    return output


def extract_ucell_column(
    result_adata: ad.AnnData,
    signature: str,
) -> np.ndarray:
    candidates = [
        f"{signature}_UCell",
        f"{signature}_resolvi_UCell",
        signature,
    ]
    source = next(
        (
            column
            for column in candidates
            if column
            in result_adata.obs.columns
        ),
        None,
    )
    if source is None:
        raise KeyError(
            "Could not locate pyUCell output "
            f"for {signature!r}. Columns: "
            f"{result_adata.obs.columns[-20:].tolist()}"
        )

    return (
        pd.to_numeric(
            result_adata.obs[source],
            errors="coerce",
        )
        .fillna(0)
        .to_numpy(
            dtype=np.float32
        )
    )


def run_ucell_compat(
    temp: ad.AnnData,
    signatures: dict[str, list[str]],
    *,
    prefer_device: str,
) -> tuple[ad.AnnData, dict]:
    if not PYUCELL_AVAILABLE:
        raise RuntimeError(
            "Fresh fallback scores are required, "
            "but pyUCell is unavailable. Run this "
            "notebook in the existing pyUCell "
            "environment."
        )

    candidate_kwargs = {
        "signatures": signatures,
        "layer": None,
        "max_rank": min(
            int(UCELL_MAX_RANK),
            temp.n_vars,
        ),
        "ties_method": (
            "min"
            if prefer_device
            != "cpu"
            else "average"
        ),
        "missing_genes": "skip",
        "chunk_size": min(
            int(
                UCELL_INNER_CHUNK_SIZE
            ),
            max(
                1,
                temp.n_obs,
            ),
        ),
        "w_neg": 1.0,
        "suffix": "_UCell",
        "n_jobs": (
            1
            if prefer_device
            != "cpu"
            else int(
                UCELL_CPU_N_JOBS
            )
        ),
        "device": prefer_device,
    }

    kwargs = {
        key: value
        for key, value
        in candidate_kwargs.items()
        if _ucell_accepts(key)
    }
    omitted = sorted(
        set(candidate_kwargs)
        - set(kwargs)
    )

    try:
        result = (
            uc.compute_ucell_scores(
                temp,
                **kwargs,
            )
        )
        if isinstance(
            result,
            ad.AnnData,
        ):
            temp = result

        return temp, {
            "backend": (
                f"device:{prefer_device}"
                if "device" in kwargs
                else "legacy_cpu_api"
            ),
            "kwargs_omitted": omitted,
        }

    except Exception as first_exc:
        if (
            prefer_device == "cpu"
            or not UCELL_GPU_FALLBACK_TO_CPU
        ):
            raise

        warnings.warn(
            "GPU pyUCell scoring failed; "
            "retrying the current chunk on CPU. "
            f"{type(first_exc).__name__}: "
            f"{first_exc}"
        )

        cpu_kwargs = dict(kwargs)
        cpu_kwargs.pop(
            "device",
            None,
        )
        if _ucell_accepts(
            "device"
        ):
            cpu_kwargs[
                "device"
            ] = "cpu"
        if "n_jobs" in cpu_kwargs:
            cpu_kwargs[
                "n_jobs"
            ] = int(
                UCELL_CPU_N_JOBS
            )
        if "ties_method" in cpu_kwargs:
            cpu_kwargs[
                "ties_method"
            ] = "average"

        result = (
            uc.compute_ucell_scores(
                temp,
                **cpu_kwargs,
            )
        )
        if isinstance(
            result,
            ad.AnnData,
        ):
            temp = result

        return temp, {
            "backend": "cpu_fallback",
            "GPU_error": (
                f"{type(first_exc).__name__}: "
                f"{first_exc}"
            ),
            "kwargs_omitted": omitted,
        }


def open_corrected_layer(
    zarr_path: Path,
):
    root = zarr.open_group(
        str(zarr_path),
        mode="r",
    )
    return (
        root["layers"][
            CORRECTED_LAYER
        ]
    )


def ensure_canonical_scores(
    sample: str,
    adata: ad.AnnData,
    paths: dict[str, Path],
) -> tuple[pd.DataFrame, dict]:
    """Reuse suitable 03C scores and compute missing/refined scores."""
    coverage_rows = []
    compute_signatures = {}
    score_key_to_signature = {}
    reused = {}

    # Standard signatures: copy an existing 03C score when available.
    for score_key, config in (
        STANDARD_SIGNATURES.items()
    ):
        desired = (
            CANONICAL_SCORE_COLUMNS[
                score_key
            ]
        )

        existing = next(
            (
                column
                for column in config[
                    "existing_columns"
                ]
                if column
                in adata.obs.columns
            ),
            None,
        )

        if (
            USE_EXISTING_03C_SCORES
            and existing is not None
        ):
            adata.obs[
                desired
            ] = (
                pd.to_numeric(
                    adata.obs[
                        existing
                    ],
                    errors="coerce",
                )
                .fillna(0)
                .to_numpy(
                    dtype=np.float32
                )
            )
            reused[
                score_key
            ] = existing
        elif (
            desired
            in adata.obs.columns
            and not
            OVERWRITE_FALLBACK_SCORES
        ):
            reused[
                score_key
            ] = desired
        else:
            signature_name = (
                f"fallback_{score_key}"
            )
            present = map_genes(
                adata.var_names,
                config["genes"],
            )
            compute_signatures[
                signature_name
            ] = present
            score_key_to_signature[
                score_key
            ] = signature_name

        present = map_genes(
            adata.var_names,
            config["genes"],
        )
        coverage_rows.append(
            {
                "score_key": score_key,
                "n_requested": len(
                    config["genes"]
                ),
                "n_present": len(
                    present
                ),
                "fraction_present": (
                    len(present)
                    / max(
                        len(
                            config[
                                "genes"
                            ]
                        ),
                        1,
                    )
                ),
                "present_genes": ";".join(
                    present
                ),
                "source": reused.get(
                    score_key,
                    "fresh_corrected_UCell",
                ),
            }
        )

    # Refined signatures: calculate fresh unless already present.
    refined_mapping = {
        "CD8_noCD8B": "CD8",
        "Treg_core": "Treg",
        "NK_specific": "NK",
        "Myeloid_macrophage_specific": (
            "Myeloid"
        ),
        "Fibroblast_collagen": (
            "Fibroblast"
        ),
    }

    for signature_name, genes in (
        REFINED_SIGNATURES.items()
    ):
        score_key = (
            refined_mapping[
                signature_name
            ]
        )
        desired = (
            CANONICAL_SCORE_COLUMNS[
                score_key
            ]
        )

        present = map_genes(
            adata.var_names,
            genes,
        )

        if (
            desired
            in adata.obs.columns
            and not
            OVERWRITE_FALLBACK_SCORES
        ):
            reused[
                score_key
            ] = desired
        else:
            internal_name = (
                f"fallback_{signature_name}"
            )
            compute_signatures[
                internal_name
            ] = present
            score_key_to_signature[
                score_key
            ] = internal_name

        coverage_rows.append(
            {
                "score_key": score_key,
                "n_requested": len(
                    genes
                ),
                "n_present": len(
                    present
                ),
                "fraction_present": (
                    len(present)
                    / max(
                        len(genes),
                        1,
                    )
                ),
                "present_genes": ";".join(
                    present
                ),
                "source": reused.get(
                    score_key,
                    "fresh_corrected_UCell",
                ),
            }
        )

    # Zero-fill signatures with no genes; do not pass them to pyUCell.
    empty_signatures = [
        name
        for name, genes
        in compute_signatures.items()
        if len(genes) == 0
    ]
    for empty in empty_signatures:
        compute_signatures.pop(
            empty
        )
        score_key = next(
            key
            for key, value
            in score_key_to_signature.items()
            if value == empty
        )
        adata.obs[
            CANONICAL_SCORE_COLUMNS[
                score_key
            ]
        ] = np.zeros(
            adata.n_obs,
            dtype=np.float32,
        )

    if not compute_signatures:
        return (
            pd.DataFrame(
                coverage_rows
            ),
            {
                "reused_scores": reused,
                "fresh_scores": [],
                "backend": "no_fresh_scores_needed",
            },
        )

    if not paths["zarr"].exists():
        raise FileNotFoundError(
            paths["zarr"]
        )

    corrected = open_corrected_layer(
        paths["zarr"]
    )
    if tuple(
        corrected.shape
    ) != tuple(
        adata.shape
    ):
        raise ValueError(
            f"{sample}: corrected Zarr "
            f"shape {corrected.shape} does "
            f"not match H5AD {adata.shape}."
        )

    if paths[
        "zarr_summary"
    ].exists():
        zarr_summary = json.loads(
            paths[
                "zarr_summary"
            ].read_text(
                encoding="utf-8"
            )
        )
        expected_obs = (
            zarr_summary.get(
                "obs_names_sha256"
            )
        )
        expected_var = (
            zarr_summary.get(
                "var_names_sha256"
            )
        )
        if (
            expected_obs
            and expected_obs
            != names_hash(
                adata.obs_names
            )
        ):
            raise ValueError(
                f"{sample}: H5AD/Zarr "
                "obs names do not align."
            )
        if (
            expected_var
            and expected_var
            != names_hash(
                adata.var_names
            )
        ):
            raise ValueError(
                f"{sample}: H5AD/Zarr "
                "var names do not align."
            )

    output_arrays = {
        score_key: np.zeros(
            adata.n_obs,
            dtype=np.float32,
        )
        for score_key
        in score_key_to_signature
        if score_key not in reused
        and CANONICAL_SCORE_COLUMNS[
            score_key
        ]
        not in adata.obs.columns
    }

    backend_records = []

    for start in range(
        0,
        adata.n_obs,
        int(
            UCELL_OUTER_CELL_CHUNK
        ),
    ):
        end = min(
            start
            + int(
                UCELL_OUTER_CELL_CHUNK
            ),
            adata.n_obs,
        )

        corrected_chunk = np.asarray(
            corrected[
                start:end,
                :,
            ],
            dtype=np.float32,
        )

        temp = ad.AnnData(
            X=corrected_chunk,
            obs=pd.DataFrame(
                index=adata.obs_names[
                    start:end
                ].copy()
            ),
            var=pd.DataFrame(
                index=adata.var_names.copy()
            ),
        )

        temp, backend = run_ucell_compat(
            temp,
            compute_signatures,
            prefer_device=PREFER_DEVICE,
        )
        backend_records.append(
            backend
        )

        for (
            score_key,
            signature_name,
        ) in score_key_to_signature.items():
            desired = (
                CANONICAL_SCORE_COLUMNS[
                    score_key
                ]
            )
            if (
                desired
                in adata.obs.columns
                and not
                OVERWRITE_FALLBACK_SCORES
            ):
                continue

            output_arrays[
                score_key
            ][
                start:end
            ] = extract_ucell_column(
                temp,
                signature_name,
            )

        del (
            temp,
            corrected_chunk,
        )
        gc.collect()

        print(
            f"{sample}: fresh UCell "
            f"{end:,}/{adata.n_obs:,}"
        )

    for (
        score_key,
        values,
    ) in output_arrays.items():
        adata.obs[
            CANONICAL_SCORE_COLUMNS[
                score_key
            ]
        ] = values

    return (
        pd.DataFrame(
            coverage_rows
        ),
        {
            "reused_scores": reused,
            "fresh_scores": sorted(
                output_arrays
            ),
            "backend_records": (
                backend_records[:3]
            ),
        },
    )

In [7]:
# ---------------------------------------------------------------------
# Robust score standardization and threshold helpers
# ---------------------------------------------------------------------
def percentile_rank(
    values: np.ndarray,
    mask: np.ndarray | None = None,
) -> np.ndarray:
    values = np.asarray(
        values,
        dtype=float,
    )

    if mask is None:
        mask = np.ones(
            len(values),
            dtype=bool,
        )
    else:
        mask = np.asarray(
            mask,
            dtype=bool,
        )

    finite = (
        mask
        & np.isfinite(values)
    )

    output = np.full(
        len(values),
        np.nan,
        dtype=np.float32,
    )

    if finite.sum() == 0:
        return output

    series = pd.Series(
        values[finite]
    )
    ranks = (
        series.rank(
            method="average",
            pct=True,
        )
        .to_numpy(
            dtype=np.float32
        )
    )
    output[finite] = ranks
    return output


def robust_z_score(
    values: np.ndarray,
    mask: np.ndarray | None = None,
) -> tuple[
    np.ndarray,
    float,
    float,
]:
    values = np.asarray(
        values,
        dtype=float,
    )

    if mask is None:
        mask = np.ones(
            len(values),
            dtype=bool,
        )
    else:
        mask = np.asarray(
            mask,
            dtype=bool,
        )

    finite = (
        mask
        & np.isfinite(values)
    )
    output = np.full(
        len(values),
        np.nan,
        dtype=np.float32,
    )

    if finite.sum() == 0:
        return (
            output,
            float("nan"),
            float("nan"),
        )

    median = float(
        np.nanmedian(
            values[finite]
        )
    )
    mad = float(
        1.4826
        * np.nanmedian(
            np.abs(
                values[finite]
                - median
            )
        )
    )

    if (
        not np.isfinite(mad)
        or mad < 1e-8
    ):
        # A zero-MAD score is common for sparse signatures.
        nonzero = values[
            finite
            & (
                values
                > median
            )
        ]
        if len(nonzero):
            scale = float(
                np.nanstd(
                    nonzero
                )
            )
        else:
            scale = 1.0

        if (
            not np.isfinite(scale)
            or scale < 1e-8
        ):
            scale = 1.0
    else:
        scale = mad

    output[
        np.isfinite(values)
    ] = (
        (
            values[
                np.isfinite(values)
            ]
            - median
        )
        / scale
    ).astype(
        np.float32
    )

    return (
        output,
        median,
        scale,
    )


def adaptive_score_threshold(
    values: np.ndarray,
    *,
    quantile: float,
    robust_z: float,
    floor: float,
    mask: np.ndarray | None = None,
) -> dict:
    values = np.asarray(
        values,
        dtype=float,
    )

    if mask is None:
        mask = np.ones(
            len(values),
            dtype=bool,
        )
    else:
        mask = np.asarray(
            mask,
            dtype=bool,
        )

    finite = (
        mask
        & np.isfinite(values)
    )

    if finite.sum() == 0:
        threshold = float(
            "inf"
        )
        quantile_cut = float(
            "nan"
        )
        median = float(
            "nan"
        )
        scale = float(
            "nan"
        )
        robust_cut = float(
            "nan"
        )
    else:
        subset = values[finite]
        quantile_cut = float(
            np.quantile(
                subset,
                float(
                    quantile
                ),
            )
        )
        _, median, scale = (
            robust_z_score(
                values,
                mask=mask,
            )
        )
        robust_cut = float(
            median
            + float(
                robust_z
            )
            * scale
        )
        threshold = max(
            float(floor),
            quantile_cut,
            robust_cut,
        )

    strong = (
        np.isfinite(values)
        & (
            values
            >= threshold
        )
    )

    z_values, _, _ = (
        robust_z_score(
            values,
            mask=mask,
        )
    )
    percentiles = (
        percentile_rank(
            values,
            mask=mask,
        )
    )

    return {
        "threshold": threshold,
        "quantile_cut": quantile_cut,
        "robust_cut": robust_cut,
        "median": median,
        "scale": scale,
        "strong": strong,
        "z": z_values,
        "percentile": percentiles,
        "n_reference_cells": int(
            finite.sum()
        ),
    }

In [8]:
# ---------------------------------------------------------------------
# Hierarchical broad-lineage and T-subtype annotation
# ---------------------------------------------------------------------
def raw_detected(
    adata: ad.AnnData,
    genes: list[str],
) -> dict[str, np.ndarray]:
    lookup = {}
    upper = {
        str(name).upper(): name
        for name in adata.var_names
    }

    for gene in genes:
        mapped = upper.get(
            gene.upper()
        )
        if mapped is None:
            lookup[
                gene
            ] = np.zeros(
                adata.n_obs,
                dtype=bool,
            )
            continue

        matrix = sp.csr_matrix(
            adata[
                :,
                [mapped],
            ].X
        )
        lookup[
            gene
        ] = (
            np.asarray(
                (
                    matrix > 0
                ).sum(
                    axis=1
                )
            )
            .ravel()
            > 0
        )

    return lookup


def annotate_sample_hierarchically(
    adata: ad.AnnData,
    sample: str,
) -> tuple[
    pd.DataFrame,
    dict,
]:
    score = {
        key: (
            pd.to_numeric(
                adata.obs[column],
                errors="coerce",
            )
            .fillna(0)
            .to_numpy(
                dtype=np.float32
            )
        )
        for key, column
        in CANONICAL_SCORE_COLUMNS.items()
    }

    # Composite broad programs.
    score[
        "Tumor"
    ] = np.maximum.reduce(
        [
            score[
                "Epithelial"
            ],
            score[
                "Melanoma_melanocytic"
            ],
            score[
                "Melanoma_dedifferentiated"
            ],
        ]
    ).astype(
        np.float32
    )
    score[
        "B_plasma"
    ] = np.maximum(
        score[
            "B_cell"
        ],
        score[
            "Plasma_cell"
        ],
    ).astype(
        np.float32
    )
    score[
        "Monocyte_macrophage"
    ] = score[
        "Myeloid"
    ]
    score[
        "Fibroblast"
    ] = score[
        "Fibroblast"
    ]
    score[
        "Endothelial"
    ] = score[
        "Endothelial"
    ]
    score[
        "NK"
    ] = score[
        "NK"
    ]
    score[
        "T_core"
    ] = score[
        "T_core"
    ]

    broad_keys = [
        "T_core",
        "Tumor",
        "Endothelial",
        "Monocyte_macrophage",
        "Fibroblast",
        "B_plasma",
        "NK",
    ]

    threshold_rows = []
    metrics = {}

    for key in broad_keys:
        config = (
            BROAD_THRESHOLD_CONFIG[
                key
            ]
        )
        result = (
            adaptive_score_threshold(
                score[key],
                **config,
            )
        )
        metrics[
            key
        ] = result

        threshold_rows.append(
            {
                "sample": sample,
                "analysis_level": (
                    "broad_lineage"
                ),
                "signature": key,
                "quantile": config[
                    "quantile"
                ],
                "robust_z_parameter": config[
                    "robust_z"
                ],
                "fixed_floor": config[
                    "floor"
                ],
                "quantile_cut": result[
                    "quantile_cut"
                ],
                "robust_cut": result[
                    "robust_cut"
                ],
                "threshold": result[
                    "threshold"
                ],
                "median": result[
                    "median"
                ],
                "scale": result[
                    "scale"
                ],
                "n_reference_cells": result[
                    "n_reference_cells"
                ],
                "n_strong": int(
                    result[
                        "strong"
                    ].sum()
                ),
            }
        )

        adata.obs[
            f"fallback_{key}_score"
        ] = score[key]
        adata.obs[
            f"fallback_{key}_z"
        ] = result["z"]
        adata.obs[
            f"fallback_{key}_percentile"
        ] = result[
            "percentile"
        ]
        adata.obs[
            f"fallback_{key}_strong"
        ] = result[
            "strong"
        ]

    # T-first broad-lineage hierarchy.
    non_t_keys = [
        "Tumor",
        "Endothelial",
        "Monocyte_macrophage",
        "Fibroblast",
        "B_plasma",
        "NK",
    ]

    non_t_z = np.column_stack(
        [
            metrics[key]["z"]
            for key in non_t_keys
        ]
    )
    non_t_z_safe = np.nan_to_num(
        non_t_z,
        nan=-np.inf,
    )
    max_non_t_z = non_t_z_safe.max(
        axis=1
    )

    t_lineage = (
        metrics[
            "T_core"
        ][
            "strong"
        ]
        & (
            metrics[
                "T_core"
            ][
                "z"
            ]
            >= (
                max_non_t_z
                + float(
                    T_LINEAGE_Z_MARGIN
                )
            )
        )
    )

    # Rank the non-T competitors for cells not accepted as T.
    dominant_index = np.argmax(
        non_t_z_safe,
        axis=1,
    )
    dominant_z = non_t_z_safe[
        np.arange(
            adata.n_obs
        ),
        dominant_index,
    ]

    if len(non_t_keys) > 1:
        sorted_z = np.sort(
            non_t_z_safe,
            axis=1,
        )
        runner_up_z = (
            sorted_z[
                :,
                -2,
            ]
        )
    else:
        runner_up_z = np.full(
            adata.n_obs,
            -np.inf,
        )

    dominant_key = np.asarray(
        [
            non_t_keys[index]
            for index
            in dominant_index
        ],
        dtype=object,
    )
    dominant_strong = np.asarray(
        [
            metrics[
                dominant_key[index]
            ][
                "strong"
            ][
                index
            ]
            for index
            in range(
                adata.n_obs
            )
        ],
        dtype=bool,
    )
    dominant_clear = (
        dominant_z
        >= (
            runner_up_z
            + float(
                NON_T_Z_MARGIN
            )
        )
    )

    broad_label = np.full(
        adata.n_obs,
        "Other_unresolved",
        dtype=object,
    )
    broad_label[
        t_lineage
    ] = "T_lineage"

    non_t_assignable = (
        ~t_lineage
        & dominant_strong
        & dominant_clear
    )
    broad_label[
        non_t_assignable
    ] = dominant_key[
        non_t_assignable
    ]

    if not LABEL_B_PLASMA:
        broad_label[
            broad_label
            == "B_plasma"
        ] = "Other_unresolved"

    if not LABEL_NK:
        broad_label[
            broad_label
            == "NK"
        ] = "Other_unresolved"

    adata.obs[
        "fallback_level1_lineage"
    ] = pd.Categorical(
        broad_label,
        categories=[
            "T_lineage",
            "Tumor",
            "Endothelial",
            "Monocyte_macrophage",
            "Fibroblast",
            "B_plasma",
            "NK",
            "Other_unresolved",
        ],
    )
    adata.obs[
        "fallback_T_lineage"
    ] = t_lineage
    adata.obs[
        "fallback_T_core_strong_but_mixed"
    ] = (
        metrics[
            "T_core"
        ][
            "strong"
        ]
        & ~t_lineage
    )
    adata.obs[
        "fallback_broad_score_margin_z"
    ] = (
        metrics[
            "T_core"
        ][
            "z"
        ]
        - max_non_t_z
    ).astype(
        np.float32
    )
    adata.obs[
        "fallback_dominant_nonT_program"
    ] = pd.Categorical(
        dominant_key
    )

    # --------------------------------------------------------------
    # T-subtype hierarchy within accepted T-lineage cells.
    # --------------------------------------------------------------
    subtype_reference = t_lineage.copy()
    if subtype_reference.sum() < 20:
        subtype_reference = np.ones(
            adata.n_obs,
            dtype=bool,
        )

    cd4_values = score["CD4"]
    cd8_values = score["CD8"]
    treg_values = score["Treg"]

    cd4_threshold = float(
        max(
            SUBTYPE_SCORE_FLOOR,
            np.quantile(
                cd4_values[
                    subtype_reference
                ],
                float(
                    CD4_SUBTYPE_QUANTILE
                ),
            ),
        )
    )
    cd8_threshold = float(
        max(
            SUBTYPE_SCORE_FLOOR,
            np.quantile(
                cd8_values[
                    subtype_reference
                ],
                float(
                    CD8_SUBTYPE_QUANTILE
                ),
            ),
        )
    )

    cd4_z, cd4_median, cd4_scale = (
        robust_z_score(
            cd4_values,
            mask=subtype_reference,
        )
    )
    cd8_z, cd8_median, cd8_scale = (
        robust_z_score(
            cd8_values,
            mask=subtype_reference,
        )
    )

    treg_result = (
        adaptive_score_threshold(
            treg_values,
            mask=subtype_reference,
            **TREG_THRESHOLD_CONFIG,
        )
    )

    cd4_positive = (
        cd4_values
        >= cd4_threshold
    )
    cd8_positive = (
        cd8_values
        >= cd8_threshold
    )

    if (
        TREG_REQUIRE_CD4_COMPATIBILITY
    ):
        cd4_compatible = (
            cd4_positive
            | (
                cd4_z >= 0
            )
        )
    else:
        cd4_compatible = np.ones(
            adata.n_obs,
            dtype=bool,
        )

    treg_call = (
        t_lineage
        & treg_result[
            "strong"
        ]
        & cd4_compatible
        & (
            treg_result[
                "z"
            ]
            >= (
                cd8_z
                + float(
                    TREG_VS_CD8_Z_MARGIN
                )
            )
        )
    )

    raw_treg = raw_detected(
        adata,
        [
            "FOXP3",
            "IL2RA",
            "CTLA4",
        ],
    )
    raw_treg_anchor = (
        raw_treg[
            "FOXP3"
        ]
        | (
            raw_treg[
                "IL2RA"
            ]
            & raw_treg[
                "CTLA4"
            ]
        )
    )

    if (
        USE_RAW_TREG_ANCHOR_FOR_FINAL_LABEL
    ):
        treg_call &= (
            raw_treg_anchor
        )

    cd8_call = (
        t_lineage
        & ~treg_call
        & cd8_positive
        & (
            cd8_z
            >= (
                cd4_z
                + float(
                    CD4_CD8_Z_MARGIN
                )
            )
        )
    )

    cd4_call = (
        t_lineage
        & ~treg_call
        & ~cd8_call
        & cd4_positive
        & (
            cd4_z
            >= (
                cd8_z
                + float(
                    CD4_CD8_Z_MARGIN
                )
            )
        )
    )

    ambiguous_cd4_cd8 = (
        t_lineage
        & ~treg_call
        & ~cd8_call
        & ~cd4_call
        & cd4_positive
        & cd8_positive
    )

    subtype = np.full(
        adata.n_obs,
        "",
        dtype=object,
    )
    subtype[
        t_lineage
    ] = "Tcell:unspecified"
    subtype[
        ambiguous_cd4_cd8
    ] = (
        "Tcell:CD4_CD8_ambiguous"
    )
    subtype[
        cd4_call
    ] = "Tcell:CD4+"
    subtype[
        cd8_call
    ] = "Tcell:CD8+"
    subtype[
        treg_call
    ] = "Tcell:Treg"

    final_label = broad_label.copy()
    final_label[
        t_lineage
    ] = subtype[
        t_lineage
    ]

    adata.obs[
        "fallback_level2_T_subtype"
    ] = pd.Categorical(
        subtype
    )
    adata.obs[
        "fallback_cell_type"
    ] = pd.Categorical(
        final_label
    )
    adata.obs[
        "fallback_Treg_signature"
    ] = treg_call
    adata.obs[
        "fallback_Treg_raw_anchor"
    ] = raw_treg_anchor
    adata.obs[
        "fallback_Treg_high_confidence"
    ] = (
        treg_call
        & raw_treg_anchor
    )
    adata.obs[
        "fallback_CD4_positive"
    ] = cd4_positive
    adata.obs[
        "fallback_CD8_positive"
    ] = cd8_positive

    for (
        key,
        values,
        z_values,
    ) in (
        (
            "CD4",
            cd4_values,
            cd4_z,
        ),
        (
            "CD8",
            cd8_values,
            cd8_z,
        ),
        (
            "Treg",
            treg_values,
            treg_result[
                "z"
            ],
        ),
    ):
        adata.obs[
            f"fallback_{key}_subtype_score"
        ] = values
        adata.obs[
            f"fallback_{key}_subtype_z"
        ] = z_values

    threshold_rows.extend(
        [
            {
                "sample": sample,
                "analysis_level": (
                    "T_subtype"
                ),
                "signature": "CD4",
                "quantile": (
                    CD4_SUBTYPE_QUANTILE
                ),
                "robust_z_parameter": np.nan,
                "fixed_floor": (
                    SUBTYPE_SCORE_FLOOR
                ),
                "quantile_cut": (
                    cd4_threshold
                ),
                "robust_cut": np.nan,
                "threshold": (
                    cd4_threshold
                ),
                "median": cd4_median,
                "scale": cd4_scale,
                "n_reference_cells": int(
                    subtype_reference.sum()
                ),
                "n_strong": int(
                    (
                        t_lineage
                        & cd4_positive
                    ).sum()
                ),
            },
            {
                "sample": sample,
                "analysis_level": (
                    "T_subtype"
                ),
                "signature": (
                    "CD8_noCD8B"
                ),
                "quantile": (
                    CD8_SUBTYPE_QUANTILE
                ),
                "robust_z_parameter": np.nan,
                "fixed_floor": (
                    SUBTYPE_SCORE_FLOOR
                ),
                "quantile_cut": (
                    cd8_threshold
                ),
                "robust_cut": np.nan,
                "threshold": (
                    cd8_threshold
                ),
                "median": cd8_median,
                "scale": cd8_scale,
                "n_reference_cells": int(
                    subtype_reference.sum()
                ),
                "n_strong": int(
                    (
                        t_lineage
                        & cd8_positive
                    ).sum()
                ),
            },
            {
                "sample": sample,
                "analysis_level": (
                    "T_subtype"
                ),
                "signature": (
                    "Treg_core"
                ),
                "quantile": (
                    TREG_THRESHOLD_CONFIG[
                        "quantile"
                    ]
                ),
                "robust_z_parameter": (
                    TREG_THRESHOLD_CONFIG[
                        "robust_z"
                    ]
                ),
                "fixed_floor": (
                    TREG_THRESHOLD_CONFIG[
                        "floor"
                    ]
                ),
                "quantile_cut": (
                    treg_result[
                        "quantile_cut"
                    ]
                ),
                "robust_cut": (
                    treg_result[
                        "robust_cut"
                    ]
                ),
                "threshold": (
                    treg_result[
                        "threshold"
                    ]
                ),
                "median": (
                    treg_result[
                        "median"
                    ]
                ),
                "scale": (
                    treg_result[
                        "scale"
                    ]
                ),
                "n_reference_cells": int(
                    subtype_reference.sum()
                ),
                "n_strong": int(
                    treg_call.sum()
                ),
            },
        ]
    )

    thresholds = pd.DataFrame(
        threshold_rows
    )

    t_count = int(
        t_lineage.sum()
    )
    treg_count = int(
        treg_call.sum()
    )
    treg_fraction = (
        treg_count
        / max(
            t_count,
            1,
        )
    )

    warnings_list = []
    if (
        t_count > 0
        and treg_fraction
        > float(
            TREG_WARNING_FRACTION_OF_TCELLS
        )
    ):
        warning = (
            f"{sample}: Treg fraction "
            f"{treg_fraction:.1%} exceeds "
            f"the configured warning level "
            f"{TREG_WARNING_FRACTION_OF_TCELLS:.1%}."
        )
        warnings.warn(warning)
        warnings_list.append(
            warning
        )

    summary = {
        "n_cells": int(
            adata.n_obs
        ),
        "n_T_lineage": t_count,
        "fraction_T_lineage": float(
            t_lineage.mean()
        ),
        "n_Treg": treg_count,
        "fraction_Treg_of_T": float(
            treg_fraction
        ),
        "n_Treg_with_raw_anchor": int(
            (
                treg_call
                & raw_treg_anchor
            ).sum()
        ),
        "warnings": warnings_list,
    }

    return (
        thresholds,
        summary,
    )

In [9]:
# ---------------------------------------------------------------------
# Optional diagnostic Leiden on the signature-score matrix
# ---------------------------------------------------------------------
def add_signature_score_leiden(
    adata: ad.AnnData,
) -> None:
    if not RUN_SIGNATURE_LEIDEN:
        return

    import scanpy as sc

    columns = [
        "fallback_T_core_z",
        "fallback_Tumor_z",
        "fallback_Endothelial_z",
        "fallback_Monocyte_macrophage_z",
        "fallback_Fibroblast_z",
        "fallback_B_plasma_z",
        "fallback_NK_z",
        "fallback_CD4_subtype_z",
        "fallback_CD8_subtype_z",
        "fallback_Treg_subtype_z",
    ]
    columns = [
        column
        for column in columns
        if column in adata.obs.columns
    ]

    matrix = (
        adata.obs[columns]
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .fillna(0)
        .to_numpy(
            dtype=np.float32
        )
    )

    score_adata = ad.AnnData(
        X=matrix,
        obs=pd.DataFrame(
            index=adata.obs_names.copy()
        ),
        var=pd.DataFrame(
            index=columns
        ),
    )

    sc.pp.neighbors(
        score_adata,
        n_neighbors=min(
            int(
                SIGNATURE_LEIDEN_N_NEIGHBORS
            ),
            max(
                2,
                score_adata.n_obs
                - 1,
            ),
        ),
        use_rep="X",
        random_state=0,
    )
    sc.tl.leiden(
        score_adata,
        resolution=float(
            SIGNATURE_LEIDEN_RESOLUTION
        ),
        key_added=(
            "fallback_signature_leiden"
        ),
        random_state=0,
    )
    sc.tl.umap(
        score_adata,
        random_state=0,
    )

    adata.obs[
        "fallback_signature_leiden"
    ] = (
        score_adata.obs[
            "fallback_signature_leiden"
        ]
        .astype(str)
        .to_numpy()
    )
    adata.obsm[
        "X_fallback_signature_umap"
    ] = np.asarray(
        score_adata.obsm[
            "X_umap"
        ],
        dtype=np.float32,
    )

In [10]:
# ---------------------------------------------------------------------
# Diagnostic plots and metadata exports
# ---------------------------------------------------------------------
def plotting_indices(
    n_obs: int,
) -> np.ndarray:
    if n_obs <= int(
        PLOT_MAX_CELLS
    ):
        return np.arange(
            n_obs
        )

    rng = np.random.default_rng(
        0
    )
    return np.sort(
        rng.choice(
            n_obs,
            size=int(
                PLOT_MAX_CELLS
            ),
            replace=False,
        )
    )


def save_count_plot(
    adata: ad.AnnData,
    sample: str,
    path: Path,
) -> None:
    counts = (
        adata.obs[
            "fallback_cell_type"
        ]
        .astype(str)
        .value_counts()
    )

    ax = counts.plot(
        kind="bar",
        figsize=(11, 6),
    )
    ax.set_title(
        f"{sample}: fallback UCell annotation"
    )
    ax.set_xlabel(
        "Cell type"
    )
    ax.set_ylabel(
        "Cells"
    )
    ax.tick_params(
        axis="x",
        rotation=35,
    )
    plt.tight_layout()
    plt.savefig(
        path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close()


def save_T_score_diagnostics(
    adata: ad.AnnData,
    sample: str,
    path: Path,
) -> None:
    indices = plotting_indices(
        adata.n_obs
    )

    t_score = pd.to_numeric(
        adata.obs[
            "fallback_T_core_score"
        ],
        errors="coerce",
    ).to_numpy(dtype=float)
    myeloid = pd.to_numeric(
        adata.obs[
            "fallback_Monocyte_macrophage_score"
        ],
        errors="coerce",
    ).to_numpy(dtype=float)
    tumor = pd.to_numeric(
        adata.obs[
            "fallback_Tumor_score"
        ],
        errors="coerce",
    ).to_numpy(dtype=float)

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14, 6),
    )

    image = axes[0].hexbin(
        t_score[indices],
        myeloid[indices],
        gridsize=90,
        mincnt=1,
    )
    axes[0].set_xlabel(
        "T-core corrected UCell"
    )
    axes[0].set_ylabel(
        "Macrophage-specific corrected UCell"
    )
    axes[0].set_title(
        "T versus myeloid"
    )
    fig.colorbar(
        image,
        ax=axes[0],
        label="Cells",
    )

    image = axes[1].hexbin(
        t_score[indices],
        tumor[indices],
        gridsize=90,
        mincnt=1,
    )
    axes[1].set_xlabel(
        "T-core corrected UCell"
    )
    axes[1].set_ylabel(
        "Tumor composite corrected UCell"
    )
    axes[1].set_title(
        "T versus tumor"
    )
    fig.colorbar(
        image,
        ax=axes[1],
        label="Cells",
    )

    fig.suptitle(
        f"{sample}: broad lineage score diagnostics"
    )
    fig.tight_layout()
    fig.savefig(
        path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_spatial_plot(
    adata: ad.AnnData,
    sample: str,
    path: Path,
) -> str | None:
    key = choose_spatial_key(
        adata
    )
    if key is None:
        return None

    coordinates = np.asarray(
        adata.obsm[key],
        dtype=float,
    )
    indices = plotting_indices(
        adata.n_obs
    )

    labels = (
        adata.obs[
            "fallback_cell_type"
        ]
        .iloc[indices]
        .astype(str)
    )
    categories = (
        labels
        .value_counts()
        .index
        .tolist()
    )

    fig, ax = plt.subplots(
        figsize=(9, 8)
    )

    for category in categories:
        mask = (
            labels.to_numpy(
                dtype=str
            )
            == category
        )
        ax.scatter(
            coordinates[
                indices[mask],
                0,
            ],
            coordinates[
                indices[mask],
                1,
            ],
            s=2,
            linewidths=0,
            alpha=0.75,
            label=category,
            rasterized=True,
        )

    ax.set_title(
        f"{sample}: fallback UCell cell types"
    )
    ax.set_xlabel("Spatial x")
    ax.set_ylabel("Spatial y")
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.legend(
        bbox_to_anchor=(
            1.02,
            1,
        ),
        loc="upper left",
        fontsize=7,
        frameon=False,
    )
    fig.tight_layout()
    fig.savefig(
        path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)

    return key

In [11]:
# ---------------------------------------------------------------------
# Process one sample
# ---------------------------------------------------------------------
def process_sample(
    sample: str,
) -> tuple[
    dict,
    ad.AnnData,
]:
    paths = paths_for_sample(
        sample
    )

    if not paths[
        "scored"
    ].exists():
        raise FileNotFoundError(
            paths["scored"]
        )

    print(
        "\n"
        + "=" * 90
    )
    print(
        "Fallback UCell annotation:",
        sample,
    )
    print(
        "Input:",
        paths["scored"],
    )

    adata = ad.read_h5ad(
        paths["scored"]
    )
    adata.obs_names = (
        adata.obs_names.astype(str)
    )

    metadata = SAMPLE_INFO[
        sample
    ]
    for column, value in (
        metadata.items()
    ):
        adata.obs[column] = value
    adata.obs[
        "sample"
    ] = sample

    resume_available = (
        RESUME_FROM_EXISTING_METADATA
        and paths["metadata"].exists()
    )

    if resume_available:
        print(
            "[stage] restore existing fallback metadata ..."
        )
        (
            coverage,
            thresholds,
            annotation_summary,
            scoring_info,
        ) = restore_existing_fallback_metadata(
            adata,
            paths,
            sample,
        )
    else:
        print(
            "[stage] calculate/reuse corrected UCell scores ..."
        )
        coverage, scoring_info = (
            ensure_canonical_scores(
                sample,
                adata,
                paths,
            )
        )
        coverage.to_csv(
            paths["coverage"],
            index=False,
        )

        print(
            "[stage] hierarchical annotation ..."
        )
        thresholds, annotation_summary = (
            annotate_sample_hierarchically(
                adata,
                sample,
            )
        )
        thresholds.to_csv(
            paths["thresholds"],
            index=False,
        )

    add_signature_score_leiden(
        adata
    )

    count_table = (
        adata.obs[
            "fallback_cell_type"
        ]
        .astype(str)
        .value_counts()
        .rename_axis(
            "fallback_cell_type"
        )
        .rename(
            "n_cells"
        )
        .reset_index()
    )
    count_table[
        "fraction"
    ] = (
        count_table[
            "n_cells"
        ]
        / max(
            adata.n_obs,
            1,
        )
    )
    count_table[
        "sample"
    ] = sample
    count_table[
        "patient"
    ] = metadata[
        "patient"
    ]
    count_table[
        "cancer_type"
    ] = metadata[
        "cancer_type"
    ]
    count_table[
        "biopsy_stage"
    ] = metadata[
        "biopsy_stage"
    ]
    count_table.to_csv(
        paths["counts"],
        index=False,
    )

    save_count_plot(
        adata,
        sample,
        paths["figures"]
        / f"{sample}_fallback_cell_type_counts.png",
    )
    save_T_score_diagnostics(
        adata,
        sample,
        paths["figures"]
        / f"{sample}_fallback_T_score_diagnostics.png",
    )
    spatial_key = save_spatial_plot(
        adata,
        sample,
        paths["figures"]
        / f"{sample}_fallback_spatial_cell_types.png",
    )

    # Compact cell-level output.
    output_columns = [
        column
        for column in adata.obs.columns
        if (
            column.startswith(
                "fallback_"
            )
            or column
            in {
                "sample",
                "patient",
                "cancer_type",
                "biopsy_stage",
            }
        )
    ]
    cell_metadata = (
        adata.obs[
            output_columns
        ]
        .copy()
    )
    cell_metadata.insert(
        0,
        "cell_id",
        adata.obs_names.astype(str),
    )

    if WRITE_METADATA_PARQUET:
        cell_metadata.to_parquet(
            paths["metadata"],
            index=False,
        )

    if (
        WRITE_SPATIAL_PARQUET
        and spatial_key
        is not None
    ):
        coordinates = np.asarray(
            adata.obsm[
                spatial_key
            ],
            dtype=np.float32,
        )
        spatial = (
            cell_metadata.copy()
        )
        spatial[
            "spatial_x"
        ] = coordinates[
            :,
            0,
        ]
        spatial[
            "spatial_y"
        ] = coordinates[
            :,
            1,
        ]
        spatial[
            "spatial_source"
        ] = spatial_key
        spatial.to_parquet(
            paths["spatial"],
            index=False,
        )

    adata.uns[
        "fallback_resolvi_ucell_annotation"
    ] = {
        "pipeline_version": (
            PIPELINE_VERSION
        ),
        "method": (
            "sample-adaptive hierarchical "
            "ResolVI-corrected UCell"
        ),
        "source_03c_h5ad": str(
            paths["scored"]
        ),
        "source_03a_corrected_zarr": str(
            paths["zarr"]
        ),
        "corrected_layer": (
            CORRECTED_LAYER
        ),
        "CD8B_used": False,
        # The v1 value contained a list of dictionaries under
        # backend_records. AnnData/HDF5 attempted to encode that object array
        # as strings and raised:
        # "Can't implicitly convert non-string objects to strings".
        "scoring_info": h5ad_safe_scoring_info(
            scoring_info
        ),
        "threshold_table": str(
            paths["thresholds"]
        ),
        "cell_type_column": (
            "fallback_cell_type"
        ),
        "broad_lineage_column": (
            "fallback_level1_lineage"
        ),
        "T_subtype_column": (
            "fallback_level2_T_subtype"
        ),
    }

    t_mask = (
        adata.obs[
            "fallback_T_lineage"
        ]
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )

    if WRITE_ANNOTATED_H5AD:
        print(
            "[stage] write full annotated H5AD ..."
        )
        safe_write_h5ad(
            adata,
            paths["annotated"],
            compression=H5AD_COMPRESSION,
        )

    if WRITE_T_ONLY_H5AD:
        print(
            "[stage] write T-only H5AD ..."
        )
        tdata = adata[
            t_mask
        ].copy()
        tdata.uns[
            "fallback_resolvi_ucell_annotation"
        ][
            "subset_interpretation"
        ] = (
            "T-lineage cells from "
            "signature hierarchy"
        )
        safe_write_h5ad(
            tdata,
            paths["T_only"],
            compression=H5AD_COMPRESSION,
        )
        del tdata

    summary = {
        "pipeline_version": (
            PIPELINE_VERSION
        ),
        "sample": sample,
        **metadata,
        "n_cells": int(
            adata.n_obs
        ),
        **annotation_summary,
        "cell_type_counts": (
            adata.obs[
                "fallback_cell_type"
            ]
            .astype(str)
            .value_counts()
            .to_dict()
        ),
        "scoring_info": (
            scoring_info
        ),
        "spatial_key": (
            spatial_key
        ),
        "annotated_h5ad": str(
            paths["annotated"]
        )
        if WRITE_ANNOTATED_H5AD
        else None,
        "T_only_h5ad": str(
            paths["T_only"]
        )
        if WRITE_T_ONLY_H5AD
        else None,
        "metadata": str(
            paths["metadata"]
        )
        if WRITE_METADATA_PARQUET
        else None,
    }
    write_json(
        summary,
        paths["summary"],
    )

    print(
        sample,
        summary[
            "cell_type_counts"
        ],
    )

    return (
        summary,
        adata,
    )

In [12]:
# ---------------------------------------------------------------------
# Run all samples
# ---------------------------------------------------------------------
import traceback

results = {}
failures = {}

for sample in SECTION_NAMES:
    try:
        summary, adata = (
            process_sample(
                sample
            )
        )
        results[
            sample
        ] = summary

        del adata
        gc.collect()

    except Exception as exc:
        failures[
            sample
        ] = (
            f"{type(exc).__name__}: "
            f"{exc}"
        )
        print(
            f"[FAILED] {sample}: "
            f"{type(exc).__name__}: "
            f"{exc}"
        )
        traceback.print_exc(
            limit=12
        )
        if not CONTINUE_ON_ERROR:
            raise

    finally:
        plt.close("all")
        gc.collect()

write_json(
    results,
    OUTPUT_ROOT
    / "all_sample_fallback_annotation_results.json",
)
write_json(
    failures,
    OUTPUT_ROOT
    / "all_sample_fallback_annotation_failures.json",
)

print(
    "Completed:",
    sorted(
        results
    ),
)
print(
    "Failures:",
    json.dumps(
        failures,
        indent=2,
    ),
)


Fallback UCell annotation: Screen_39_21
Input: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/03c_reference_ucell_rescue/Screen_39_21/Screen_39_21_reference_ucell_scored.h5ad
[stage] restore existing fallback metadata ...
Screen_39_21: restored 59 fallback columns from existing Parquet; fresh UCell scoring was skipped.
[stage] write full annotated H5AD ...
Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/08_fallback_ucell_hierarchical_annotation/Screen_39_21/Screen_39_21_fallback_ucell_annotated.h5ad
[stage] write T-only H5AD ...
Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/08_fallback_ucell_hierarchical_annotation/Screen_39_21/Screen_39_21_fallback_Tcells_only.h5ad
Screen_39_21 {'Other_unresolved': 13909, 'NK': 2143, 'Tumor': 2086, 'B_plasma': 2068, 'Endothelial': 1630, 'Tcell:unspecified': 396, 'Tcell:CD4+': 3

In [13]:
# ---------------------------------------------------------------------
# Cross-sample count summaries and quick comparison plots
# ---------------------------------------------------------------------
summary_rows = []
cell_type_rows = []

for sample, result in (
    results.items()
):
    summary_rows.append(
        {
            "sample": sample,
            "patient": result[
                "patient"
            ],
            "cancer_type": result[
                "cancer_type"
            ],
            "biopsy_stage": result[
                "biopsy_stage"
            ],
            "n_cells": result[
                "n_cells"
            ],
            "n_T_lineage": result[
                "n_T_lineage"
            ],
            "fraction_T_lineage": result[
                "fraction_T_lineage"
            ],
            "n_Treg": result[
                "n_Treg"
            ],
            "fraction_Treg_of_T": result[
                "fraction_Treg_of_T"
            ],
            "n_Treg_with_raw_anchor": result[
                "n_Treg_with_raw_anchor"
            ],
        }
    )

    for (
        label,
        count,
    ) in result[
        "cell_type_counts"
    ].items():
        cell_type_rows.append(
            {
                "sample": sample,
                "patient": result[
                    "patient"
                ],
                "cancer_type": result[
                    "cancer_type"
                ],
                "biopsy_stage": result[
                    "biopsy_stage"
                ],
                "fallback_cell_type": label,
                "n_cells": int(
                    count
                ),
            }
        )

sample_summary = pd.DataFrame(
    summary_rows
)
cell_type_summary = pd.DataFrame(
    cell_type_rows
)

sample_summary.to_csv(
    OUTPUT_ROOT
    / "fallback_annotation_summary_by_sample.csv",
    index=False,
)
cell_type_summary.to_csv(
    OUTPUT_ROOT
    / "fallback_cell_type_counts_by_sample.csv",
    index=False,
)

if len(sample_summary):
    ax = sample_summary.set_index(
        "sample"
    )[
        [
            "n_T_lineage",
            "n_Treg",
            "n_Treg_with_raw_anchor",
        ]
    ].plot(
        kind="bar",
        figsize=(13, 7),
        width=0.80,
    )
    ax.set_title(
        "Fallback UCell T-lineage and Treg counts"
    )
    ax.set_xlabel("Sample")
    ax.set_ylabel("Cells")
    ax.tick_params(
        axis="x",
        rotation=35,
    )
    plt.tight_layout()
    plt.savefig(
        OUTPUT_ROOT
        / "fallback_T_and_Treg_counts_by_sample.png",
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close()

manifest = {
    "pipeline_version": (
        PIPELINE_VERSION
    ),
    "method": (
        "sample-adaptive hierarchical "
        "ResolVI-corrected UCell"
    ),
    "n_samples_completed": int(
        len(results)
    ),
    "n_samples_failed": int(
        len(failures)
    ),
    "sample_summary": str(
        OUTPUT_ROOT
        / "fallback_annotation_summary_by_sample.csv"
    ),
    "cell_type_counts": str(
        OUTPUT_ROOT
        / "fallback_cell_type_counts_by_sample.csv"
    ),
    "CD8B_used": False,
    "failures": failures,
}
write_json(
    manifest,
    OUTPUT_ROOT
    / "fallback_annotation_manifest.json",
)

display(sample_summary)
display(cell_type_summary)

,sample,patient,cancer_type,biopsy_stage,n_cells,n_T_lineage,fraction_T_lineage,n_Treg,fraction_Treg_of_T,n_Treg_with_raw_anchor
0,Screen_39_21,patient_39_21,NSCLC,Screen,22950,1114,0.048540,19,0.017056,1
1,C2D15_39_21,patient_39_21,NSCLC,C2D15,10822,1079,0.099704,8,0.007414,0
2,Screen_17_26,patient_17_26,NSCLC,Screen,47896,0,0.000000,0,0.000000,0
3,C2D15_17_26,patient_17_26,NSCLC,C2D15,87913,0,0.000000,0,0.000000,0
4,Screen_18_23,patient_18_23,melanoma,Screen,72386,147,0.002031,0,0.000000,0
5,C2D15_18_23,patient_18_23,melanoma,C2D15,18594,34,0.001829,0,0.000000,0
6,Screen_16_22,patient_16_22,melanoma,Screen,70438,76,0.001079,0,0.000000,0
7,C2D15_16_22,patient_16_22,melanoma,C2D15,76633,171,0.002231,0,0.000000,0
8,Screen_30_16,patient_30_16,melanoma,Screen,66977,1690,0.025233,42,0.024852,6
9,C2D15_30_16,patient_30_16,melanoma,C2D15,351799,0,0.000000,0,0.000000,0


,sample,patient,cancer_type,biopsy_stage,fallback_cell_type,n_cells
0,Screen_39_21,patient_39_21,NSCLC,Screen,Other_unresolved,13909
1,Screen_39_21,patient_39_21,NSCLC,Screen,NK,2143
2,Screen_39_21,patient_39_21,NSCLC,Screen,Tumor,2086
3,Screen_39_21,patient_39_21,NSCLC,Screen,B_plasma,2068
4,Screen_39_21,patient_39_21,NSCLC,Screen,Endothelial,1630
...,...,...,...,...,...,...
100,C2D15_23_25,patient_23_25,colon_cancer,C2D15,Tcell:CD8+,32
101,C2D15_23_25,patient_23_25,colon_cancer,C2D15,Tcell:CD4+,21
102,C2D15_23_25,patient_23_25,colon_cancer,C2D15,NK,2
103,C2D15_23_25,patient_23_25,colon_cancer,C2D15,Tcell:Treg,1


# Interpretation and rapid tuning

## Primary columns

```python
fallback_cell_type
fallback_level1_lineage
fallback_level2_T_subtype
fallback_T_lineage
```

Treg audit:

```python
fallback_Treg_signature
fallback_Treg_raw_anchor
fallback_Treg_high_confidence
```

## Recommended immediate review

For each sample, inspect:

```text
<sample>_fallback_cell_type_counts.csv
<sample>_fallback_ucell_thresholds.csv
figures/<sample>_fallback_spatial_cell_types.png
figures/<sample>_fallback_T_score_diagnostics.png
```

Then review the cross-sample table:

```text
fallback_annotation_summary_by_sample.csv
```

## Making T lineage more sensitive

Reduce:

```python
BROAD_THRESHOLD_CONFIG["T_core"]["quantile"]
# 0.90 → 0.85
```

or:

```python
T_LINEAGE_Z_MARGIN
# 0.25 → 0.10
```

The margin is usually the safer parameter to relax first.

## Making T lineage more specific

Increase:

```python
T_LINEAGE_Z_MARGIN
# 0.25 → 0.50
```

or increase the T-core quantile.

## Making Treg more conservative

Increase:

```python
TREG_THRESHOLD_CONFIG["quantile"]
# 0.975 → 0.99
```

or:

```python
TREG_VS_CD8_Z_MARGIN
# 0.50 → 0.75
```

For the most conservative output:

```python
USE_RAW_TREG_ANCHOR_FOR_FINAL_LABEL = True
```

The signature-only and raw-anchored flags are saved regardless.

## CD8 interpretation

`CD8B` is not used anywhere in the fallback CD8 signature. The CD8 call relies
on:

```text
T-core lineage gate
CD8A/cytotoxic corrected UCell
CD8 program dominance over CD4
```

Because cytotoxic genes can also be high in NK cells, NK-specific UCell remains
a competing broad-lineage program.

## No cluster dependence

The final labels are cell-level and do not depend on Leiden. The optional
signature-score Leiden column is a diagnostic grouping only.